In [ ]:
# %% [markdown]
# # SHL Recommendation Engine - Evaluation

# %%
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from app.recommender import SHLRecommender

# %%
# Load train data (create this from the provided dataset)
# Format: query, ground_truth_urls (as list)
train_data = [
    {
        'query': 'I am hiring for Java developers who can also collaborate effectively with my business teams.',
        'ground_truth': [
            'https://www.shl.com/solutions/products/java-skills-test/',
            'https://www.shl.com/solutions/products/collaboration-assessment/'
        ]
    },
    # Add more from your provided train set
]

train_df = pd.DataFrame(train_data)

# %%
# Initialize recommender
recommender = SHLRecommender('../data/shl_catalog.csv')

# %%
def calculate_recall_at_k(recommended, ground_truth, k=10):
    """Calculate Recall@K"""
    if len(ground_truth) == 0:
        return 0
    
    recommended_set = set(recommended[:k])
    ground_truth_set = set(ground_truth)
    
    relevant_found = len(recommended_set & ground_truth_set)
    return relevant_found / len(ground_truth_set)

# %%
# Evaluate
results = []

for idx, row in train_df.iterrows():
    query = row['query']
    ground_truth = row['ground_truth']
    
    # Get recommendations
    recommendations = recommender.recommend(query, top_k=10)
    recommended_urls = [r['url'] for r in recommendations]
    
    # Calculate metrics
    recall_10 = calculate_recall_at_k(recommended_urls, ground_truth, k=10)
    
    results.append({
        'query': query[:50] + '...',
        'recall@10': recall_10,
        'num_recommendations': len(recommendations),
        'num_ground_truth': len(ground_truth)
    })

# %%
# Display results
results_df = pd.DataFrame(results)
print(results_df)

# %%
# Calculate mean recall
mean_recall = results_df['recall@10'].mean()
print(f"\nMean Recall@10: {mean_recall:.3f}")

# %%
# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x=results_df.index, y='recall@10')
plt.title('Recall@10 by Query')
plt.xlabel('Query Index')
plt.ylabel('Recall@10')
plt.ylim(0, 1)
plt.show()

# %%
# Generate test set predictions
test_queries = [
    "Looking for Python developers with SQL knowledge",
    "Need cognitive ability tests for graduate roles",
    # Add all test queries from provided dataset
]

predictions = recommender.batch_recommend(test_queries)
predictions.to_csv('../submission.csv', index=False)
print(f"Generated {len(predictions)} predictions")